In [1]:
# ==============================================================================
# CELL 1: INSTALL & CLONE
# ==============================================================================

!pip install -q transformers bitsandbytes accelerate datasets peft

REPO = "https://github.com/yonghaoliang/cs175-Mars-text2sql.git"
REPO_DIR = "/content/cs175-Mars-text2sql"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.insert(0, f"{REPO_DIR}/colabtesting")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.2 MB/s eta 0:00:00
Cloning into '/content/cs175-Mars-text2sql'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 58 (delta 12), reused 52 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 58.78 KiB | 19.59 MiB/s, done.
Resolving deltas: 100% (12/12), done.


In [2]:
# ==============================================================================
# CELL 2: LOAD MODELS
# ==============================================================================

import torch
import warnings
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
warnings.filterwarnings('ignore')

MODEL_NAME = "defog/sqlcoder-7b-2"
print(f"🚀 Loading {MODEL_NAME}...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
print("✅ SQLCoder loaded!")

judge_model_id = "NousResearch/Meta-Llama-3-8B-Instruct"
print(f"📥 Loading {judge_model_id} as AI Judge...")
bnb_config_judge = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
judge_tokenizer = AutoTokenizer.from_pretrained(judge_model_id)
judge_tokenizer.pad_token = judge_tokenizer.eos_token
judge_model = AutoModelForCausalLM.from_pretrained(judge_model_id, device_map="auto", quantization_config=bnb_config_judge)
print("✅ Llama-3 Judge loaded!")

import sys; sys.path.insert(0, "/content/cs175-Mars-text2sql/colabtesting")
import Models
Models.model          = model
Models.tokenizer      = tokenizer
Models.judge_model     = judge_model
Models.judge_tokenizer = judge_tokenizer
print("✅ Models injected into Models module!")

🚀 Loading defog/sqlcoder-7b-2...


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

✅ Llama-3 Judge loaded!
✅ Models injected into Models module!


In [3]:
# ==============================================================================
# CELL 3: DATASET & DATABASE SETUP
# ==============================================================================

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from datasets import load_dataset
import sys; sys.path.insert(0, "/content/cs175-Mars-text2sql/colabtesting")
from Helpers import set_data_root, get_db_path, execute_query

# 1. Load Spider questions (no DB download — DBs come from GitHub)
print("⏳ Loading Spider questions from HuggingFace...")
dataset = load_dataset('spider', split='validation')
test_df = pd.DataFrame(dataset).copy()
test_df = test_df.iloc[[9, 202, 588]]
print(f"✅ Loaded {len(test_df)} questions: indices {test_df.index.tolist()}")

# 2. Point helpers at the 3testingdata folder in the cloned repo
set_data_root(f"{REPO_DIR}/3testingdata")

# 3. Sanity check all 3 databases
print("\n🔎 Checking database files:")
all_ok = True
for db_id in test_df['db_id'].unique():
    path = get_db_path(db_id)
    ok, res = execute_query("SELECT name FROM sqlite_master WHERE type='table';", path)
    status = f"✅ tables: {list(res.iloc[:,0])}" if ok else f"❌ {res}"
    print(f"  {db_id:20s} -> {status}")
    if not ok:
        all_ok = False

if not all_ok:
    raise RuntimeError("❌ One or more databases are missing — check 3testingdata/ in GitHub.")

output_dir = "/content/evaluation_results"
os.makedirs(output_dir, exist_ok=True)
print(f"\n💾 Results will be saved to: {output_dir}")

⏳ Loading Spider questions from HuggingFace...


README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

✅ Loaded 3 questions: indices [9, 202, 588]

🔎 Checking database files:
  concert_singer       -> ✅ tables: ['stadium', 'singer', 'concert', 'singer_in_concert']
  flight_2             -> ✅ tables: ['airlines', 'airports', 'flights']
  tvshow               -> ✅ tables: ['TV_Channel', 'TV_series', 'Cartoon']

💾 Results will be saved to: /content/evaluation_results


In [4]:
# ==============================================================================
# CELL 4: RUN ALL 4 METHODS
# ==============================================================================

from tqdm.auto import tqdm
import sys; sys.path.insert(0, "/content/cs175-Mars-text2sql/colabtesting")
from Methods import (
    generate_sql_baseline,
    generate_sql_fewshot,
    generate_sql_cot,
    generate_sql_refinement,
)
import sys; sys.path.insert(0, "/content/cs175-Mars-text2sql/colabtesting")
from Helpers import get_schema_from_sqlite

baseline_results, fewshot_results, cot_results, refinement_results = [], [], [], []

for idx, (original_index, row) in enumerate(test_df.iterrows()):
    db_id    = row['db_id']
    question = row['question']
    gold_sql = row['query']
    db_path  = get_db_path(db_id)
    schema   = get_schema_from_sqlite(db_path)

    print(f"\n{'='*50}")
    print(f"🔹 Q ({idx+1}/{len(test_df)}): {question}")
    print(f"⭐ Gold: {gold_sql}")

    base_sql = generate_sql_baseline(question, schema)
    baseline_results.append({"ID": original_index, "Question": question, "Gold_SQL": gold_sql, "Predicted_SQL": base_sql, "db_id": db_id})
    print(f"🤖 [Baseline]:  {base_sql}")

    fs_sql = generate_sql_fewshot(question, schema)
    fewshot_results.append({"ID": original_index, "Question": question, "Gold_SQL": gold_sql, "Predicted_SQL": fs_sql, "db_id": db_id})
    print(f"🤖 [Few-Shot]:  {fs_sql}")

    cot_sql = generate_sql_cot(question, schema)
    cot_results.append({"ID": original_index, "Question": question, "Gold_SQL": gold_sql, "Predicted_SQL": cot_sql, "db_id": db_id})
    print(f"🤖 [CoT]:       {cot_sql}")

    attempts = generate_sql_refinement(question, schema, db_path)
    refinement_results.append({
        "ID": original_index, "Question": question, "Gold_SQL": gold_sql,
        "Attempt_1_SQL": attempts[0], "Attempt_2_SQL": attempts[1],
        "Attempt_3_SQL": attempts[2], "Final_Predicted_SQL": attempts[-1],
        "db_id": db_id
    })
    print(f"🔄 [Refinement Attempt 1]: {attempts[0]}")
    print(f"🔄 [Refinement Attempt 2]: {attempts[1]}")
    print(f"🔄 [Refinement Attempt 3]: {attempts[2]}")

pd.DataFrame(baseline_results).to_csv(f"{output_dir}/baseline_results_final.csv",   index=False)
pd.DataFrame(fewshot_results).to_csv(f"{output_dir}/fewshot_results_final.csv",     index=False)
pd.DataFrame(cot_results).to_csv(f"{output_dir}/cot_results_final.csv",             index=False)
pd.DataFrame(refinement_results).to_csv(f"{output_dir}/refinement_results_multi_attempt_final.csv", index=False)
print("\n✅ All generation methods finished and saved!")


🔹 Q (1/3): What are  the different countries with singers above age 20?
⭐ Gold: SELECT DISTINCT country FROM singer WHERE age  >  20
🤖 [Baseline]:  SELECT s.country FROM singer s WHERE s.age > 20 GROUP BY s.country ORDER BY COUNT(s.country) DESC NULLS LAST;
🤖 [Few-Shot]:  SELECT s.country FROM singer s WHERE s.age > 20 GROUP BY s.country;
🤖 [CoT]:       SELECT s.country FROM singer s WHERE s.age > 20;
🔄 [Refinement Attempt 1]: SELECT s.country FROM singer s WHERE s.age > 20 GROUP BY s.country;
🔄 [Refinement Attempt 2]: SELECT s.country FROM singer s WHERE s.age > 20 GROUP BY s.country;
🔄 [Refinement Attempt 3]: SELECT s.country FROM singer s WHERE s.age > 20 GROUP BY s.country;

🔹 Q (2/3): What are the names of airports in Aberdeen?
⭐ Gold: SELECT AirportName FROM AIRPORTS WHERE City = "Aberdeen"
🤖 [Baseline]:  SELECT a.AirportName FROM airports a WHERE a.City ILIKE '%Aberdeen%'
🤖 [Few-Shot]:  SELECT AirportName FROM airports WHERE City ILIKE '%Aberdeen%';
🤖 [CoT]:       SELECT a.Airp

In [5]:
# ==============================================================================
# CELL 5: GRADE BASELINE / FEW-SHOT / COT
# ==============================================================================

import sys; sys.path.insert(0, "/content/cs175-Mars-text2sql/colabtesting")
from Evaluate import grade_csv_with_execution, calculate_accuracy
import pandas as pd

files_to_grade = [
    (f"{output_dir}/baseline_results_final.csv",  f"{output_dir}/baseline_results_EXEC_SCORED.csv"),
    (f"{output_dir}/fewshot_results_final.csv",   f"{output_dir}/fewshot_results_EXEC_SCORED.csv"),
    (f"{output_dir}/cot_results_final.csv",       f"{output_dir}/cot_results_EXEC_SCORED.csv"),
]

for input_file, output_file in files_to_grade:
    total_pts, max_pts, acc = grade_csv_with_execution(input_file, output_file)
    method_name = input_file.split('/')[-1].split('_')[0].upper()

    df = pd.read_csv(output_file)
    print(f"\n========================================")
    print(f"📋 {method_name} — SCORES PER QUESTION")
    print(f"========================================")
    for _, row in df.iterrows():
        print(f"Q: {row['Question']}")
        print(f"   Score: {int(row['AI_Score'])}/10  |  {row['AI_Reason']}")
    print(f"----------------------------------------")
    print(f"   Total: {total_pts} / {max_pts} ({acc:.2f}%)")
    print(f"========================================")

Grading baseline_results_final.csv:   0%|          | 0/3 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



📋 BASELINE — SCORES PER QUESTION
Q: What are  the different countries with singers above age 20?
   Score: 8/10  |  The student's SQL query is functionally equivalent to the gold standard, but the execution result is slightly off due to the GROUP BY and ORDER BY clauses, which are not present in the gold standard.
Q: What are the names of airports in Aberdeen?
   Score: 1/10  |  Major errors, execution failed with SQLite errors, or returns completely wrong data. The student's SQL query contains a syntax error due to the incorrect use of ILIKE operator.
Q: What are the names of all cartoons directed by Ben Jones?
   Score: 2/10  |  The student's SQL query is close, but it's missing the 'Directed_by' column and the table name is incorrect. The correct table name is 'Cartoon', not 'cartoon'. Additionally, the join condition is incorrect, it should be 'c.Directed_by = tc.series_name'.
----------------------------------------
   Total: 11 / 30 (36.67%)


Grading fewshot_results_final.csv:   0%|          | 0/3 [00:00<?, ?it/s]


📋 FEWSHOT — SCORES PER QUESTION
Q: What are  the different countries with singers above age 20?
   Score: 8/10  |  The student's SQL query is functionally equivalent to the gold standard, but it's missing the DISTINCT keyword, which is necessary to get the unique countries. The query executes successfully and returns the correct data, but it's missing the distinctness.
Q: What are the names of airports in Aberdeen?
   Score: 1/10  |  Major errors, execution failed with SQLite errors, or returns completely wrong data. The student's SQL query contains a syntax error due to the use of ILIKE which is not a valid SQLite function.
Q: What are the names of all cartoons directed by Ben Jones?
   Score: 6/10  |  The student's SQL query is functionally correct and executes successfully, but it only returns an empty result set because the condition `tc.Country = 'United States'` filters out all cartoons, whereas the gold standard query returns cartoons directed by Ben Jones regardless of the cha

Grading cot_results_final.csv:   0%|          | 0/3 [00:00<?, ?it/s]


📋 COT — SCORES PER QUESTION
Q: What are  the different countries with singers above age 20?
   Score: 9/10  |  The student's SQL query is functionally equivalent to the gold standard, but the result has duplicate rows because the student forgot to use the DISTINCT keyword.
Q: What are the names of airports in Aberdeen?
   Score: 1/10  |  Major errors, execution failed with SQLite errors, or returns completely wrong data. The student's SQL query contains a syntax error due to the use of ILIKE, which is not a valid SQLite operator. The correct operator to use is LIKE or =.
Q: What are the names of all cartoons directed by Ben Jones?
   Score: 2/10  |  The student's SQL query is close, but it's missing the 'Directed_by' column and the table name is incorrect. The correct table name is 'Cartoon', not 'cartoon'. Additionally, the join condition is incorrect, it should be 'c.Directed_by = tc.series_name'.
----------------------------------------
   Total: 12 / 30 (40.00%)


In [6]:
# ==============================================================================
# CELL 6: GRADE REFINEMENT ATTEMPTS
# ==============================================================================

import sys; sys.path.insert(0, "/content/cs175-Mars-text2sql/colabtesting")
from Evaluate import grade_multi_attempts
import pandas as pd

input_file  = f"{output_dir}/refinement_results_multi_attempt_final.csv"
output_file = f"{output_dir}/refinement_COMPARED_SCORED.csv"

avg1, avg2, avg3 = grade_multi_attempts(input_file, output_file)

df = pd.read_csv(output_file)
print("\n========================================")
print("📈 REFINEMENT SCORES PER QUESTION")
print("========================================")
for _, row in df.iterrows():
    print(f"Q: {row['Question']}")
    print(f"   Attempt 1: {int(row['Score_1'])}/10")
    print(f"   Attempt 2: {int(row['Score_2'])}/10")
    print(f"   Attempt 3: {int(row['Score_3'])}/10")
print("========================================")
print(f"🔹 Avg Attempt 1: {avg1:.2f}")
print(f"🔹 Avg Attempt 2: {avg2:.2f}")
print(f"🔹 Avg Attempt 3: {avg3:.2f}")
print(f"🚀 Total Improvement: {avg3 - avg1:+.2f} points")
print("========================================")

print("\n🎉 ALL TASKS COMPLETED SUCCESSFULLY! 🎉")


🚀 Grading 3 attempts for 3 questions...


Grading All Attempts:   0%|          | 0/3 [00:00<?, ?it/s]


📈 REFINEMENT SCORES PER QUESTION
Q: What are  the different countries with singers above age 20?
   Attempt 1: 8/10
   Attempt 2: 8/10
   Attempt 3: 8/10
Q: What are the names of airports in Aberdeen?
   Attempt 1: 1/10
   Attempt 2: 1/10
   Attempt 3: 1/10
Q: What are the names of all cartoons directed by Ben Jones?
   Attempt 1: 6/10
   Attempt 2: 10/10
   Attempt 3: 10/10
🔹 Avg Attempt 1: 5.00
🔹 Avg Attempt 2: 6.33
🔹 Avg Attempt 3: 6.33
🚀 Total Improvement: +1.33 points

🎉 ALL TASKS COMPLETED SUCCESSFULLY! 🎉
